In [1]:
import pickle
import pandas as pd

import torch
import torch.nn.functional as F

from tqdm import tqdm


In [2]:
# Import classes
from utils.config import load_hyperparameters
from core import SimpleTransformer
from utils.data import AlphabetDataset, collate


In [3]:
# Import HYPERPARAMETERS
hyperparams = load_hyperparameters("./HYPERPARAMETERS.toml")

PROJECT_NAME = hyperparams['PROJECT_NAME']
PROJECT_VERSION = hyperparams['PROJECT_VERSION']
MODEL_NAME = hyperparams['MODEL_NAME']
VOCAB_SIZE = hyperparams['VOCAB_SIZE']
EMB_DIM = hyperparams['EMB_DIM']
MAX_SEQ_LEN = hyperparams['MAX_SEQ_LEN']
LEARNING_RATE = hyperparams['LEARNING_RATE']
BATCH_SIZE = hyperparams['BATCH_SIZE']
SEED = hyperparams['SEED']
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


torch.manual_seed(SEED)
torch.device(hyperparams['DEVICE'])


device(type='mps')

In [4]:
data = pd.read_parquet("./data/tokenized_alphabet_sequences.parquet")
id2token = pickle.load(open("./data/alphabet_id2token.pkl", "rb"))
token2id = pickle.load(open("./data/alphabet_token2id.pkl", "rb"))


In [5]:
print("id for token 'a':", token2id['a'])
print("token for id 1:", id2token[1])


id for token 'a': 1
token for id 1: a


In [6]:
data.head()

,sequence,actual,mask,example
0,nopqrstuv,"[14, 15, 16, 17, 18, 19, 20, 21, 22]","[1, 0, 1, 1, 1, 1, 1, 1, 0]","[14, 0, 16, 17, 18, 19, 20, 21, 0]"
1,abcd,"[1, 2, 3, 4]","[1, 1, 1, 1]","[1, 2, 3, 4]"
2,bcdefgh,"[2, 3, 4, 5, 6, 7, 8]","[1, 1, 1, 1, 1, 1, 1]","[2, 3, 4, 5, 6, 7, 8]"
3,stu,"[19, 20, 21]","[1, 1, 1]","[19, 20, 21]"
4,qrst,"[17, 18, 19, 20]","[1, 1, 0, 1]","[17, 18, 0, 20]"


In [7]:
tiny_data = data.iloc[:10]
dataset = AlphabetDataset(tiny_data, VOCAB_SIZE)



In [8]:
dataloader = torch.utils.data.DataLoader(dataset, 
                                         batch_size=1, 
                                         shuffle=False,
                                        collate_fn=collate)

In [9]:

# After creating dataloader
print("Checking first batch:")
a, x, m = next(iter(dataloader))
print("Input sequence (x):", x)
print("Target sequence (a):", a)
print("Mask:", m)
print("\nDecoded input:", ''.join([id2token[i.item()] for i in x[0]]))
print("Decoded target:", ''.join([id2token[i.item()] for i in a[0]]))
# A spot check on this — looks ok!


Checking first batch:
Input sequence (x): tensor([[14, 15, 16, 17, 18, 19, 20,  0, 22]])
Target sequence (a): tensor([[14, 15, 16, 17, 18, 19, 20, 21, 22]])
Mask: tensor([[True, True, True, True, True, True, True, True, True]])

Decoded input: nopqrst<UNK>v
Decoded target: nopqrstuv


In [10]:
print("\nToken mappings:")
print("UNK token id:", token2id['<UNK>'])
print("First few token mappings:")
for token, idx in list(token2id.items())[:5]:
    print(f"{token}: {idx}")


Token mappings:
UNK token id: 0
First few token mappings:
<UNK>: 0
a: 1
b: 2
c: 3
d: 4


## Prepare the model

In [11]:
model = SimpleTransformer(VOCAB_SIZE, EMB_DIM, num_blocks=5)
model.to(DEVICE)
optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [12]:
# Trainin loop
# Add wandb logging
import wandb
wandb.init(project=PROJECT_NAME, name=f"{PROJECT_NAME}-{PROJECT_VERSION}-{MODEL_NAME}")
model.train()
# Debug prints

for epoch in tqdm(range(1500)):
    for i, (actual, ex, mask) in enumerate(dataloader):
        # Move to device and get model output
        actual = actual.to(DEVICE)
        ex = ex.to(DEVICE)
        mask = mask.to(DEVICE)
        logits = model(ex, mask) 

        # Reshape for loss computation
        logits = logits.view(-1, VOCAB_SIZE)
        actual = actual.view(-1)

        # ✨ TODO: Need to only compute loss on masked tokens, not the entire sequence
        # Compute loss
        loss = F.cross_entropy(logits, actual)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        wandb.log({"loss": loss})

    # Validate every batch
    if epoch % 100 == 0:
        model.eval()
        with torch.no_grad():
            outputs = model(ex, mask)
            pred = torch.argmax(outputs[0], dim=-1)
            act = actual.view(ex.shape[0], -1)[0]
            
            # Print probabilities for debugging
            probs = F.softmax(outputs[0], dim=-1)
            top_probs, top_tokens = torch.topk(probs, 3, dim=-1)
            
            pred_tokens = ''.join([id2token[i.item()] for i in pred])
            actual_tokens = ''.join([id2token[i.item()] for i in act])
            
            print(f"Predicted: {pred_tokens}")
            print(f"Actual:    {actual_tokens}")
            print("\nTop 3 predictions for first position:")
            for prob, tok in zip(top_probs[0], top_tokens[0]):
                print(f"{id2token[tok.item()]}: {prob:.3f}")

    model.train()
wandb.finish()


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: johnx (machine-learning-institute). Use `wandb login --relogin` to force relogin


  0%|          | 3/1500 [00:10<1:05:34,  2.63s/it]

Predicted: eee
Actual:    jkl

Top 3 predictions for first position:
e: 0.040
n: 0.040
p: 0.040


  7%|▋         | 103/1500 [00:15<01:21, 17.24it/s]

Predicted: jjj
Actual:    jkl

Top 3 predictions for first position:
j: 0.057
k: 0.055
s: 0.052


...

Predicted: lll
Actual:    jkl

Top 3 predictions for first position:
l: 0.328
j: 0.328
k: 0.326


 94%|█████████▎| 1403/1500 [01:37<00:05, 16.61it/s]

Predicted: kkk
Actual:    jkl

Top 3 predictions for first position:
k: 0.354
j: 0.327
l: 0.289


100%|██████████| 1500/1500 [01:45<00:00, 14.27it/s]


loss,█▆▅▅▄▆▃▅▄▃▆▇▅▅▅▃▂▁▂▂▄▃▅▅▁▄▃▂▅▄▅▂▂▂▁▂▂▄▅▄
loss,1.10965


In [78]:
a, x, m = next(iter(dataloader))


In [ ]:
def validate_predictions(model, dataloader, id2token, n_examples=15):
    model.eval()
    with torch.no_grad():
        # Get one batch
        actual, example, mask = next(iter(dataloader))
        
        # Get predictions
        logits = model(example, mask)  # [batch, seq, vocab]
        
        # For each example in batch (up to n_examples)
        for i in range(min(n_examples, len(actual))):
            # Get single sequence
            seq_probs = logits[i]  # # apply softmax???
            seq_actual = actual[i]  # [seq]
            
            # Get predicted tokens
            predicted_indices = torch.argmax(seq_probs, dim=-1)  # [seq]
            
            # Convert to letters
            pred_tokens = [id2token[idx.item()] for idx in predicted_indices]
            actual_tokens = [id2token[idx.item()] for idx in seq_actual]
            
            print(f"\nExample {i+1}:")
            print(f"Predicted: {''.join(pred_tokens)}")
            print(f"Actual:    {''.join(actual_tokens)}")

# Run validation
validate_predictions(model, dataloader, id2token)

In [51]:
a, x, m = next(iter(dataloader))